# Mistral Fine‑Tuning Workflow

In [ ]:

# MODELS SELECTION AND LOADING
# --------------------------------------------------
!pip install --no-deps xformers trl peft accelerate bitsandbytes

import os, datetime, torch, pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# --- Load quantised Mixtral‑8x7B ---
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
model_id = "mistralai/Mixtral-8x7B-Instruct-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Loading quantised Mixtral‑8x7B …")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    use_auth_token=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id
model.eval()
print("✅ model loaded on →", model.hf_device_map)

# --------------------------------------------------
# INFERENCE BEFORE FINE‑TUNING
# --------------------------------------------------
testing_df = pd.read_excel("/kaggle/input/dataset-completo/dataset.xlsx")[["Sentence", "TRADUZIONE ITALIANA"]]
testing_ds = Dataset.from_pandas(testing_df)
test_ds = testing_ds.train_test_split(test_size=0.05)["test"]

def translate_sample(idx: int):
    archaic = test_ds[idx]["Sentence"]
    modern_true = test_ds[idx]["TRADUZIONE ITALIANA"]
    prompt = f"Sei un esperto linguista... Traduci: {archaic}"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    modern_pred = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"[SRC] {archaic}\n[REF] {modern_true}\n[PRED] {modern_pred}\n")

for i in range(3):
    translate_sample(i)

# --------------------------------------------------
# LoRA FINE‑TUNING
# --------------------------------------------------
print("\nPreparing LoRA...")
model = prepare_model_for_kbit_training(model)
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

train_df = pd.read_excel("/kaggle/input/reduced/dataset_augmented - Copia.xlsx")[["Sentence", "TRADUZIONE ITALIANA"]]
train_ds = Dataset.from_pandas(train_df)

def preprocess(examples):
    inputs = tokenizer(examples["Sentence"], max_length=512, padding="max_length", truncation=True)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["TRADUZIONE ITALIANA"], max_length=512, padding="max_length", truncation=True)
    inputs["labels"] = labels["input_ids"]
    return inputs

train_ds = train_ds.map(preprocess, batched=True)

args = Seq2SeqTrainingArguments(
    output_dir="./results_mixtral",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    learning_rate=2e-5,
    logging_steps=25,
    save_total_limit=3,
    predict_with_generate=True,
    bf16=torch.cuda.is_available(),
    report_to="none",
)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, tokenizer=tokenizer)
print("\n▶ Fine‑tuning ...")
trainer.train()
print("✅ Fine‑tuning complete!")

# --------------------------------------------------
# INFERENCE AFTER FINE‑TUNING
# --------------------------------------------------
print("\nPost‑training translations:")
for i in range(3):
    translate_sample(i)
